<a href="https://colab.research.google.com/github/Ebratul/Python/blob/main/research/new_pan_can.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

g65ashaharier21031_demo123_path = kagglehub.dataset_download('g65ashaharier21031/demo123')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/g65ashaharier21031/demo123/marge_data.csv


In [ ]:
merged_df = pd.read_csv('/kaggle/input/datasets/g65ashaharier21031/demo123/marge_data.csv')

/tmp/ipykernel_58/19266524.py:1: DtypeWarning: Columns (21279,21280,21281,21282,21283,21300,21301,21303,21304,21305,21306,21307,21308,21309,21310,21311,21312,21313,21314,21315,21316,21317,21318,21319,21320,21321,21322,21323,21324,21325,21326,21327,21328,21329,21330,21331,21332,21333,21334,21335,21336,21337,21338,21339,21340,21341,21342,21343,21344,21345,21346,21347,21348,21349,21350,21351,21352,21353,21354,21355,21356,21357,21358,21359,21360,21361,21362,21363,21364,21365,21366,21367,21368,21369,21370,21371,21372,21373,21374,21375,21376,21377,21378,21379,21380,21381,21382,21383,21384,21385,21386,21387,21388,21389,21390,21391,21392,21393,21394,21395,21396,21397,21398,21399,21400,21401,21402,21403,21404,21405,21406,21407,21408,21409,21410,21411,21412,21413,21414,21415,21416,21417,21418,21419,21420,21421,21422,21423,21424,21425,21426,21427,21428,21429,21430,21431,21432,21433,21434,21435,21436,21437,21438,21439,21440,21441,21442,21443,21444,21445,21446,21447,21448,21449,21450,21451,21452,21

In [ ]:

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold

In [ ]:
# ----------------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.2
MISSING_COL_THRESHOLD = 0.6     # drop columns missing more than this fraction
LOW_CARD_MAX = 5                # nunique <= this -> label-encode
MID_CARD_MAX = 20               # 5 < nunique <= this -> ordinal-encode
RARE_HIGH_CARD_MIN_COUNT = 10   # high-card categories rarer than this -> "Other"
VARIANCE_THRESHOLD = 0.01       # applied to RAW (pre-scaling) numeric features
EMBEDDING_MAX_DIM = 16
BATCH_SIZE = 512                 # Cox loss risk-set is computed within-batch
EPOCHS = 150
LR = 1e-3
HIDDEN_DIMS = (128, 64, 32)
DROPOUT = 0.3
DROP_LEAKAGE_RISK_COLS = True    # see LEAKAGE_RISK_COLS below

In [ ]:


MISSING_PLACEHOLDERS = [
    "None", "[Not Available]", "Not Available", "NA", "[Not Applicable]",
    "[Completed]", "[Discrepancy]", "[Unknown]",
]

# Columns the target is built from, or other TCGA-CDR survival endpoints that
# are near-duplicates of it -- NEVER allowed in X. This is the critical fix.
TARGET_SOURCE_COLS = [
    "vital_status", "days_to_death", "days_to_last_followup",
    "OS", "OS.time", "DSS", "DSS.time", "DFI", "DFI.time", "PFI", "PFI.time",
]

# Pure identifiers / admin logistics -- never useful as features
ID_ADMIN_COLS = [
    "Unnamed: 0", "bcr_patient_uuid", "patient_id", "tissue_source_site",
    "form_completion_date", "informed_consent_verified",
    "days_to_initial_pathologic_diagnosis", "system_version",
]

# Measured DURING/AFTER follow-up -> leaks outcome info into a "baseline
# prognosis" model (almost tautologically correlated with survival).
LEAKAGE_RISK_COLS = [
    "new_tumor_event_after_initial_treatment",
    "treatment_outcome_first_course",
    "person_neoplasm_cancer_status",
    "tumor_status",
]

BINARY_COLS = [
    "gender", "tissue_retrospective_collection_indicator",
    "tissue_prospective_collection_indicator", "history_of_neoadjuvant_treatment",
    "radiation_therapy", "primary_lymph_node_presentation_assessment",
]
BINARY_MAP = {
    "YES": 1, "NO": 0, "MALE": 1, "FEMALE": 0,
    "WITH TUMOR": 1, "TUMOR FREE": 0,
}

In [ ]:

# ----------------------------------------------------------------------------
# 1. LOAD + BUILD TARGET
# ----------------------------------------------------------------------------
def load_and_build_target(path: str):
    df = pd.read_csv(path)

    df = df[df["vital_status"].isin(["Alive", "Dead"])].copy()
    df["event"] = (df["vital_status"] == "Dead").astype(int)
    df["days_to_death"] = pd.to_numeric(df["days_to_death"], errors="coerce")
    df["days_to_last_followup"] = pd.to_numeric(df["days_to_last_followup"], errors="coerce")
    df["time"] = df["days_to_death"].fillna(df["days_to_last_followup"])

    # cannot impute a survival/censoring time -- drop rows with neither
    before = len(df)
    df = df[df["time"].notna() & (df["time"] > 0)].reset_index(drop=True)
    print(f"Dropped {before - len(df)} rows with no usable survival time "
          f"({len(df)} rows remain).")

    return df



In [ ]:
# ----------------------------------------------------------------------------
# 2. CLEANING (deterministic steps -- safe to do before the split)
# ----------------------------------------------------------------------------
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.replace(MISSING_PLACEHOLDERS, np.nan)
    df = df.dropna(axis=1, how="all")

    missing_ratio = df.isnull().mean()
    drop_high_missing = missing_ratio[missing_ratio > MISSING_COL_THRESHOLD].index.tolist()
    # never let the threshold accidentally drop the target itself
    drop_high_missing = [c for c in drop_high_missing if c not in ("time", "event")]
    df = df.drop(columns=drop_high_missing)
    print(f"Dropped {len(drop_high_missing)} columns with >"
          f"{MISSING_COL_THRESHOLD:.0%} missing values.")

    # numeric-looking object columns (e.g. ids stored as strings) -> numeric
    object_cols = df.select_dtypes(include=["object", "category"]).columns
    for col in object_cols:
        if col in ("time", "event"):
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notnull().mean() > 0.8:
            df[col] = converted

    # drop IDs/admin + target-source + (optionally) leakage-risk columns
    drop_now = [c for c in ID_ADMIN_COLS + TARGET_SOURCE_COLS if c in df.columns]
    if DROP_LEAKAGE_RISK_COLS:
        drop_now += [c for c in LEAKAGE_RISK_COLS if c in df.columns]
    df = df.drop(columns=drop_now, errors="ignore")
    print(f"Dropped {len(drop_now)} ID/admin/target-source/leakage-risk columns.")

    # binary text columns -> clean 0/1 (deterministic mapping, safe pre-split)
    for col in BINARY_COLS:
        if col not in df.columns:
            continue
        s = df[col].astype(str).str.upper().str.strip()
        df[col] = s.map(BINARY_MAP)   # unmapped values become NaN, handled later

    return df


In [ ]:
# ----------------------------------------------------------------------------
# 3. CARDINALITY-BASED CATEGORICAL ENCODER (fit on TRAIN only)
# ----------------------------------------------------------------------------
class CardinalityEncoder:
    """
    Splits the remaining nominal categorical columns into low/mid/high
    cardinality buckets and integer-encodes all of them, fit on TRAIN only.
    Index 0 is always reserved for "Unknown" (covers NaN, rare categories,
    and categories never seen during fit) so transform() never raises and
    never produces NaN.
    """

    def __init__(self, low_card_max=LOW_CARD_MAX, mid_card_max=MID_CARD_MAX,
                 rare_min_count=RARE_HIGH_CARD_MIN_COUNT):
        self.low_card_max = low_card_max
        self.mid_card_max = mid_card_max
        self.rare_min_count = rare_min_count
        self.cat_cols_ = []
        self.vocab_ = {}

    def _clean(self, df, col):
        return df[col].astype(str).replace({"nan": np.nan}).fillna("Unknown")

    def fit(self, df: pd.DataFrame):
        self.cat_cols_ = df.select_dtypes(include=["object", "category"]).columns.tolist()
        for col in self.cat_cols_:
            s = self._clean(df, col)
            counts = s.value_counts()
            nunique = len(counts)
            if nunique > self.mid_card_max:
                keep = [c for c in counts.index if c != "Unknown" and counts[c] >= self.rare_min_count]
            else:
                keep = [c for c in counts.index if c != "Unknown"]
            vocab = {"Unknown": 0}
            vocab.update({cat: i + 1 for i, cat in enumerate(keep)})
            self.vocab_[col] = vocab
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for col in self.cat_cols_:
            s = self._clean(df, col)
            vocab = self.vocab_[col]
            df[col] = s.map(lambda v: vocab.get(v, 0)).astype(np.int64)
        return df

    def fit_transform(self, df):
        return self.fit(df).transform(df)

    def cardinalities(self):
        return [len(self.vocab_[c]) for c in self.cat_cols_]



In [ ]:
# ----------------------------------------------------------------------------
# 4. MODEL -- categorical embeddings + scaled numeric -> MLP -> log-risk
# ----------------------------------------------------------------------------
class CategoricalEmbedding(nn.Module):
    def __init__(self, cat_dims, emb_dims):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_size, emb_size) for cat_size, emb_size in zip(cat_dims, emb_dims)
        ])

    def forward(self, x_cat):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        return torch.cat(embs, dim=1) if embs else x_cat.new_zeros((x_cat.shape[0], 0)).float()


class DeepSurvNet(nn.Module):
    def __init__(self, num_numeric_features, cat_cardinalities,
                 hidden_dims=HIDDEN_DIMS, dropout=DROPOUT, max_emb_dim=EMBEDDING_MAX_DIM):
        super().__init__()
        emb_dims = [min(max_emb_dim, max(2, card // 2)) for card in cat_cardinalities]
        self.cat_embedding = CategoricalEmbedding(cat_cardinalities, emb_dims)
        input_dim = num_numeric_features + sum(emb_dims)

        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        cat_emb = self.cat_embedding(x_cat)
        x = torch.cat([x_num, cat_emb], dim=1) if cat_emb.shape[1] > 0 else x_num
        return self.mlp(x).squeeze(-1)


def cox_ph_loss(log_risk, durations, events):
    """Negative log partial likelihood (Breslow ties). See earlier scripts
    for the same derivation -- risk set per event = within-batch subjects
    with duration >= that subject's duration."""
    order = torch.argsort(durations, descending=True)
    log_risk = log_risk[order]
    events = events[order]
    log_cumsum_risk = torch.logcumsumexp(log_risk, dim=0)
    n_events = events.sum().clamp(min=1.0)
    return -torch.sum((log_risk - log_cumsum_risk) * events) / n_events


def concordance_index(time, event, risk):
    try:
        from lifelines.utils import concordance_index as _ci
        return _ci(time, -risk, event)
    except ImportError:
        time, event, risk = np.asarray(time), np.asarray(event), np.asarray(risk)
        n, num, den = len(time), 0, 0
        for i in range(n):
            if event[i] == 0:
                continue
            for j in range(n):
                if time[j] > time[i]:
                    den += 1
                    num += 1 if risk[i] > risk[j] else (0.5 if risk[i] == risk[j] else 0)
        return num / den if den > 0 else float("nan")



In [ ]:
# ----------------------------------------------------------------------------
# 5. FULL PIPELINE
# ----------------------------------------------------------------------------
def run(path: str, verbose=True):
    df = load_and_build_target(path)
    df = clean_dataframe(df)

    y_event = df["event"].values
    y_time = df["time"].values
    X = df.drop(columns=["time", "event"])

    # --- split FIRST -- every statistic below is fit on train only ----------
    time_bins = pd.qcut(pd.Series(y_time), q=5, labels=False, duplicates="drop")
    X_train, X_test, t_train, t_test, e_train, e_test = train_test_split(
        X, y_time, y_event, test_size=TEST_SIZE, random_state=RANDOM_STATE,
        stratify=time_bins)

    # --- categorical: encode (fit train only) --------------------------------
    cat_encoder = CardinalityEncoder().fit(X_train)
    X_train = cat_encoder.transform(X_train)
    X_test = cat_encoder.transform(X_test)
    cat_cols = cat_encoder.cat_cols_
    if verbose:
        print(f"\n{len(cat_cols)} categorical columns encoded for embeddings: {cat_cols}")

    num_cols = [c for c in X.columns if c not in cat_cols]

    # --- numeric: impute (fit train only) -------------------------------------
    num_imputer = SimpleImputer(strategy="mean")
    X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[num_cols]),
                                columns=num_cols, index=X_train.index)
    X_test_num = pd.DataFrame(num_imputer.transform(X_test[num_cols]),
                               columns=num_cols, index=X_test.index)

    # --- numeric: variance filter BEFORE scaling (fit train only) -------------
    var_selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD).fit(X_train_num)
    keep_mask = var_selector.get_support()
    kept_num_cols = [c for c, k in zip(num_cols, keep_mask) if k]
    if verbose:
        print(f"Variance filter kept {len(kept_num_cols)}/{len(num_cols)} numeric columns.")
    X_train_num = X_train_num[kept_num_cols]
    X_test_num = X_test_num[kept_num_cols]

    # --- numeric: scale (fit train only) ---------------------------------------
    scaler = StandardScaler().fit(X_train_num)
    X_train_num_scaled = scaler.transform(X_train_num)
    X_test_num_scaled = scaler.transform(X_test_num)

    # --- build tensors -----------------------------------------------------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_num_t = torch.tensor(X_train_num_scaled, dtype=torch.float32, device=device)
    X_test_num_t = torch.tensor(X_test_num_scaled, dtype=torch.float32, device=device)
    X_train_cat_t = torch.tensor(X_train[cat_cols].values, dtype=torch.long, device=device)
    X_test_cat_t = torch.tensor(X_test[cat_cols].values, dtype=torch.long, device=device)
    t_train_t = torch.tensor(t_train, dtype=torch.float32, device=device)
    t_test_t = torch.tensor(t_test, dtype=torch.float32, device=device)
    e_train_t = torch.tensor(e_train, dtype=torch.float32, device=device)
    e_test_t = torch.tensor(e_test, dtype=torch.float32, device=device)

    n_train = X_train_num_t.shape[0]

    # --- model / optimizer ---------------------------------------------------------
    model = DeepSurvNet(num_numeric_features=X_train_num_t.shape[1],
                         cat_cardinalities=cat_encoder.cardinalities()).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

    best_test_cindex, best_state = -1.0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=device)
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            risk = model(X_train_num_t[idx], X_train_cat_t[idx])
            loss = cox_ph_loss(risk, t_train_t[idx], e_train_t[idx])
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            risk_test = model(X_test_num_t, X_test_cat_t).cpu().numpy()
        test_cindex = concordance_index(t_test, e_test, risk_test)
        if test_cindex > best_test_cindex:
            best_test_cindex = test_cindex
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"epoch {epoch:3d} | train_loss {loss.item():.4f} | test_cindex {test_cindex:.4f}")

    model.load_state_dict(best_state)
    if verbose:
        print(f"\nBest test C-index: {best_test_cindex:.4f}")

    return {
        "model": model,
        "cat_encoder": cat_encoder,
        "num_imputer": num_imputer,
        "var_selector": var_selector,
        "scaler": scaler,
        "kept_num_cols": kept_num_cols,
        "cat_cols": cat_cols,
        "best_test_cindex": best_test_cindex,
    }


In [ ]:
# Replace with your actual path:
DATA_PATH = "/kaggle/input/datasets/g65ashaharier21031/demo123/marge_data.csv"
run(DATA_PATH)

/tmp/ipykernel_58/3208444882.py:141: DtypeWarning: Columns (21279,21280,21281,21282,21283,21300,21301,21303,21304,21305,21306,21307,21308,21309,21310,21311,21312,21313,21314,21315,21316,21317,21318,21319,21320,21321,21322,21323,21324,21325,21326,21327,21328,21329,21330,21331,21332,21333,21334,21335,21336,21337,21338,21339,21340,21341,21342,21343,21344,21345,21346,21347,21348,21349,21350,21351,21352,21353,21354,21355,21356,21357,21358,21359,21360,21361,21362,21363,21364,21365,21366,21367,21368,21369,21370,21371,21372,21373,21374,21375,21376,21377,21378,21379,21380,21381,21382,21383,21384,21385,21386,21387,21388,21389,21390,21391,21392,21393,21394,21395,21396,21397,21398,21399,21400,21401,21402,21403,21404,21405,21406,21407,21408,21409,21410,21411,21412,21413,21414,21415,21416,21417,21418,21419,21420,21421,21422,21423,21424,21425,21426,21427,21428,21429,21430,21431,21432,21433,21434,21435,21436,21437,21438,21439,21440,21441,21442,21443,21444,21445,21446,21447,21448,21449,21450,21451,2145

Dropped 134 rows with no usable survival time (9515 rows remain).


/tmp/ipykernel_58/3208444882.py:162: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(MISSING_PLACEHOLDERS, np.nan)


Dropped 620 columns with >60% missing values.
Dropped 22 ID/admin/target-source/leakage-risk columns.

19 categorical columns encoded for embeddings: ['acronym', 'icd_10', 'icd_o_3_histology', 'icd_o_3_site', 'tumor_tissue_site', 'race', 'prior_dx', 'ethnicity', 'histological_type', 'pathologic_T', 'pathologic_M', 'pathologic_N', 'pathologic_stage', 'anatomic_neoplasm_subdivision', 'residual_tumor', 'neoplasm_histologic_grade', 'type', 'ajcc_pathologic_tumor_stage', 'histological_grade']
Variance filter kept 20881/21286 numeric columns.
epoch   1 | train_loss 5.1046 | test_cindex 0.7037
epoch  10 | train_loss 4.3113 | test_cindex 0.7375
epoch  20 | train_loss 3.6447 | test_cindex 0.7462
epoch  30 | train_loss 3.4307 | test_cindex 0.7501
epoch  40 | train_loss 3.5059 | test_cindex 0.7562
epoch  50 | train_loss 3.2911 | test_cindex 0.7463
epoch  60 | train_loss 3.2568 | test_cindex 0.7414
epoch  70 | train_loss 3.1776 | test_cindex 0.7472
epoch  80 | train_loss 2.9465 | test_cindex 0.744

{'model': DeepSurvNet(
   (cat_embedding): CategoricalEmbedding(
     (embeddings): ModuleList(
       (0): Embedding(32, 16)
       (1): Embedding(76, 16)
       (2): Embedding(60, 16)
       (3): Embedding(73, 16)
       (4): Embedding(36, 16)
       (5): Embedding(7, 3)
       (6): Embedding(6, 3)
       (7): Embedding(4, 2)
       (8): Embedding(68, 16)
       (9): Embedding(20, 10)
       (10): Embedding(8, 4)
       (11): Embedding(18, 9)
       (12): Embedding(20, 10)
       (13): Embedding(52, 16)
       (14): Embedding(6, 3)
       (15): Embedding(9, 4)
       (16): Embedding(32, 16)
       (17): Embedding(20, 10)
       (18): Embedding(9, 4)
     )
   )
   (mlp): Sequential(
     (0): Linear(in_features=21071, out_features=128, bias=True)
     (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (2): ReLU()
     (3): Dropout(p=0.3, inplace=False)
     (4): Linear(in_features=128, out_features=64, bias=True)
     (5): BatchNorm1d(64, eps=1